# Week 35, Day 2: Observer

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w035d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
listeners = []

def on_goal(handler):
    listeners.append(handler)

def goal(team, minute):
    for handler in listeners:
        handler(team, minute)

on_goal(lambda team, minute: print(f"Scoreboard: {team} {minute}'"))
on_goal(lambda team, minute: print(f"Commentary: GOAL for {team}!"))
goal("Enyimba", 23)
on_goal(lambda team, minute: print("Stats: goals +1"))
goal("Rangers", 71)

In [ ]:
from typing import Protocol

class PriceObserver(Protocol):
    def price_changed(self, symbol: str, old: int, new: int) -> None: ...

class Stock:
    def __init__(self, symbol, price):
        self.symbol = symbol
        self._price = price
        self._observers = []

    def attach(self, observer):
        self._observers.append(observer)

    def detach(self, observer):
        self._observers.remove(observer)

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, new):
        old, self._price = self._price, new
        if new != old:
            for o in list(self._observers):
                o.price_changed(self.symbol, old, new)

class Display:
    def price_changed(self, symbol, old, new):
        print(f"{symbol}: {old} -> {new}")

class Alert:
    def __init__(self, below):
        self.below = below
        self.fired = []

    def price_changed(self, symbol, old, new):
        if new < self.below <= old:
            self.fired.append(f"{symbol} fell below {self.below}")

s = Stock("DANGCEM", 480)
alert = Alert(450)
s.attach(Display())
s.attach(alert)
for p in [470, 470, 440, 455]:
    s.price = p
print(alert.fired)

In [ ]:
class Thermostat:
    def __init__(self, celsius):
        self._celsius = celsius
        self._handlers = []

    def subscribe(self, handler):
        self._handlers.append(handler)
        return lambda: self._handlers.remove(handler)

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value != self._celsius:
            old, self._celsius = self._celsius, value
            for h in list(self._handlers):
                h(old, value)

log = []
t = Thermostat(19.0)
stop_logging = t.subscribe(lambda old, new: log.append(new))
t.subscribe(lambda old, new: print("heating on") if new < 18 else None)
t.celsius = 18.5
t.celsius = 17.5
stop_logging()
t.celsius = 21
print(log)

In [ ]:
class EventBus:
    def __init__(self):
        self._handlers = {}

    def on(self, event, handler):
        self._handlers.setdefault(event, []).append(handler)

    def emit(self, event, **data):
        handlers = list(self._handlers.get(event, []))
        for h in handlers:
            h(**data)
        return len(handlers)

bus = EventBus()
bus.on("order_paid", lambda code, pence: print(f"email receipt for {code}"))
bus.on("order_paid", lambda code, pence: print(f"stock reserved for {code}"))
bus.on("order_shipped", lambda code, **_: print(f"text customer: {code} is on its way"))

print(bus.emit("order_paid", code="A17", pence=4500))
print(bus.emit("order_shipped", code="A17", carrier="DHL"))
print(bus.emit("order_refunded", code="A17"))

In [ ]:
class Signal:
    def __init__(self):
        self._handlers = []

    def connect(self, handler):
        self._handlers.append(handler)

    def emit(self, *args):
        errors = []
        for h in list(self._handlers):
            try:
                h(*args)
            except Exception as err:
                errors.append((getattr(h, "__name__", repr(h)), err))
        return errors

def ok(x):
    print("ok got", x)

def broken(x):
    return 1 / x

s = Signal()
for h in (ok, broken, ok):
    s.connect(h)
print(s.emit(0))

> **Key idea.** Observer (Gang of Four): a subject keeps a list of observers and notifies them when its state changes, so the subject doesn't depend on who's listening and new observers need no edits to it. In Python, observers are usually callables; `subscribe` can return a function that unsubscribes. An event bus carries named events with keyword data between publishers and subscribers that don't know each other. Update state before notifying, notify only on real changes, loop over a copy of the observers, decide what a failing observer should do, and don't rely on the order of observers.

### Exercise: A scoreboard that announces

Write `Match(home, away)`, a subject with `subscribe(handler)` (returning a function that unsubscribes that handler) and `goal(team)`, which adds a goal to `"home"` or `"away"` (anything else raises `ValueError`) and then calls every handler with the new score as a tuple `(home_goals, away_goals)`. A property `score` gives the current tuple.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_match():\n    \"notifying\"\n    M = ns[\"Match\"]\n    m = M(\"Rangers\", \"Enyimba\")\n    seen, also = [], []\n    stop = m.subscribe(seen.append)\n    m.subscribe(lambda s: also.append(s == m.score))\n    m.goal(\"home\")\n    m.goal(\"away\")\n    stop()\n    m.goal(\"home\")\n    assert seen == [(1, 0), (1, 1)] and m.score == (2, 1) and also == [True, True, True], \"notify after updating\"\n    try:\n        m.goal(\"Rangers\")\n        assert False, \"a bad side should raise ValueError\"\n    except ValueError:\n        pass\n    assert m.score == (2, 1)")

### Exercise: An event bus

Write an `EventBus` with:

- `on(event, handler)`: subscribe a handler to an event name; returns a function that unsubscribes it;
- `once(event, handler)`: like `on`, but the handler is removed after it's called the first time;
- `emit(event, **data)`: calls the event's handlers in the order subscribed, each with the data as keyword arguments, and returns how many were called. A handler that unsubscribes itself or others during an `emit` doesn't affect who's called in that `emit`;
- `count(event)`: how many handlers an event has.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bus():\n    \"events\"\n    B = ns[\"EventBus\"]\n    b = B()\n    got = []\n    off = b.on(\"paid\", lambda code, pence: got.append((\"receipt\", code)))\n    b.once(\"paid\", lambda code, pence: got.append((\"first-time bonus\", code)))\n    b.on(\"shipped\", lambda code: got.append((\"shipped\", code)))\n    assert b.emit(\"paid\", code=\"A1\", pence=500) == 2 and b.emit(\"paid\", code=\"A2\", pence=100) == 1\n    assert b.emit(\"shipped\", code=\"A1\") == 1 and b.emit(\"nobody\", x=1) == 0\n    assert got == [(\"receipt\", \"A1\"), (\"first-time bonus\", \"A1\"), (\"receipt\", \"A2\"), (\"shipped\", \"A1\")]\n    off()\n    assert b.count(\"paid\") == 0 and b.emit(\"paid\", code=\"A3\", pence=1) == 0 and b.count(\"shipped\") == 1\n    calls = []\n    def a(**_):\n        calls.append(\"a\")\n        stop_b()\n    stop_a = b.on(\"x\", a)\n    stop_b = b.on(\"x\", lambda **_: calls.append(\"b\"))\n    assert b.emit(\"x\") == 2 and calls == [\"a\", \"b\"] and b.emit(\"x\") == 1")

### Exercise: An observable value

Write a class `Observable(value)` holding one value, with:

- `subscribe(handler)`: handlers are called as `handler(old, new)` whenever the value changes (setting the same value doesn't notify); returns an unsubscribe function;
- a property `value`, which can be read and set;
- `map(func)`: returns a **new** `Observable` whose value is always `func` of this one's value: it starts as `func(value)` and is updated whenever this one changes (and so notifies its own subscribers, if the result changed).

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_observable():\n    \"observing\"\n    O = ns[\"Observable\"]\n    price = O(1000)\n    seen = []\n    stop = price.subscribe(lambda old, new: seen.append((old, new)))\n    price.value = 1200\n    price.value = 1200\n    with_vat = price.map(lambda p: p * 6 // 5)\n    vat_seen = []\n    with_vat.subscribe(lambda old, new: vat_seen.append(new))\n    price.value = 1500\n    stop()\n    price.value = 1501\n    assert seen == [(1000, 1200), (1200, 1500)] and with_vat.value == 1801 and vat_seen == [1800, 1801]\n    parity = price.map(lambda p: p % 2)\n    flips = []\n    parity.subscribe(lambda old, new: flips.append(new))\n    price.value = 1503\n    price.value = 1504\n    assert flips == [0] and parity.value == 0")

### Exercise: Notifying when some fail

Write a class `Notifier` with `subscribe(handler)` and `notify(message)`. `notify` calls every handler with the message, even if some raise exceptions, and returns a list of `(handler_name, error_text)` pairs for the ones that failed, in order: the name is the handler's `__name__` (or its `repr` if it has none) and the error text is like `ZeroDivisionError: division by zero`. Handlers that succeed aren't listed.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_safe():\n    \"carrying on\"\n    N = ns[\"Notifier\"]\n    got = []\n    def email(m):\n        got.append((\"email\", m))\n    def sms(m):\n        raise ConnectionError(\"network down\")\n    def log(m):\n        got.append((\"log\", m))\n    class Broken:\n        def __call__(self, m):\n            return {}[m]\n        def __repr__(self):\n            return \"Broken()\"\n    n = N()\n    for h in (email, sms, log, Broken()):\n        n.subscribe(h)\n    errors = n.notify(\"parcel A17 delivered\")\n    assert got == [(\"email\", \"parcel A17 delivered\"), (\"log\", \"parcel A17 delivered\")]\n    assert errors == [(\"sms\", \"ConnectionError: network down\"), (\"Broken()\", \"KeyError: 'parcel A17 delivered'\")], f\"got {errors}\"\n    assert N().notify(\"x\") == []")

### Bug hunt: Told too early

The display reads the account's balance when told about a deposit, but it shows the balance from before the deposit: this prints `balance 100` instead of `balance 150`. Fix `deposit`.

In [ ]:
class Account:
    def __init__(self, balance):
        self.balance = balance
        self.observers = []

    def deposit(self, amount):
        for o in self.observers:
            o(self)
        self.balance += amount

acc = Account(100)
acc.observers.append(lambda a: print("balance", a.balance))
acc.deposit(50)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"updated first\"\n    expect_output(\"balance 150\")")

### Bug hunt: Skipped by a goodbye

The first handler unsubscribes itself after one call, and that makes the loop skip the handler after it: this prints only `once` instead of `once` then `always`. Fix `emit` so every handler that was subscribed when `emit` started is called.

In [ ]:
class Signal:
    def __init__(self):
        self.handlers = []

    def emit(self):
        for h in self.handlers:
            h()

s = Signal()
def once():
    print("once")
    s.handlers.remove(once)
s.handlers.append(once)
s.handlers.append(lambda: print("always"))
s.emit()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"nobody skipped\"\n    expect_output(\"once\\nalways\")")

### Bug hunt: Two receipts

The checkout's setup code runs twice by mistake, so the receipt handler is subscribed twice and each order sends two receipts: this prints `receipt A17` twice. A handler subscribed more than once should only be called once per event. Fix `subscribe` so this prints `receipt A17` once.

In [ ]:
class Checkout:
    def __init__(self):
        self.handlers = []

    def subscribe(self, handler):
        self.handlers.append(handler)

    def pay(self, code):
        for h in list(self.handlers):
            h(code)

def receipt(code):
    print("receipt", code)

c = Checkout()
for _ in range(2):
    c.subscribe(receipt)
c.pay("A17")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"once\"\n    expect_output(\"receipt A17\")")

### Puzzle: A subject and its observers

Arrange the lines into a door that tells its observers when it opens. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
log = []

class Subject:
    def __init__(self):
        self.hs = []
        self.v = 0
    def set(self, v):
        if v != self.v:
            self.v = v
            for h in list(self.hs):
                h(v)

s = Subject()
s.hs.append(lambda v: log.append(v * 10))
s.set(1)
s.set(1)
s.set(2)
s.hs.append(lambda v: log.append(-v))
s.set(3)

p = ___   # log
q = ___   # len(s.hs)
r = ___   # s.v

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [10, 20, 30, -3], \"p\"\n    assert (q, r) == (2, 3)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: cells that update themselves

Build a tiny spreadsheet with the Observer pattern: cells notify the formulas that depend on them, and formulas recompute themselves.

- `Cell(value)`: holds a value, which can be read and set through a property `value`. Setting a **different** value notifies its subscribers. `subscribe(handler)` registers a handler called as `handler(cell)` (the cell that changed), and returns an unsubscribe function.
- `Formula(func, *inputs)`: a kind of `Cell` whose value is `func(*input values)`. It works out its value when made, subscribes to each input, and recomputes whenever any input changes; if the recomputed value differs, it notifies its own subscribers (so formulas can depend on formulas). Setting a `Formula`'s value directly raises `AttributeError`.
- `history(cell)`: a helper that subscribes to a cell and returns a list that collects every new value the cell takes.

A formula should end up with the right value however inputs are chained, including when two paths lead to it from the same cell.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cells():\n    \"cells and formulas\"\n    C, F = ns[\"Cell\"], ns[\"Formula\"]\n    price, qty = C(450), C(2)\n    total = F(lambda p, q: p * q, price, qty)\n    vat = F(lambda t: t * 20 // 100, total)\n    gross = F(lambda t, v: t + v, total, vat)\n    h = history(gross)\n    assert (total.value, vat.value, gross.value) == (900, 180, 1080)\n    qty.value = 3\n    assert (total.value, vat.value, gross.value) == (1350, 270, 1620)\n    price.value = 500\n    assert gross.value == 1800 and h[-1] == 1800\n    qty.value = 3\n    assert h[-1] == 1800 and all(isinstance(v, int) for v in h)\n    try:\n        total.value = 1\n        assert False, \"a formula's value can't be set\"\n    except AttributeError:\n        pass\ndef test_changes_only():\n    \"notifications only on change\"\n    C, F = ns[\"Cell\"], ns[\"Formula\"]\n    n = C(4)\n    even = F(lambda x: x % 2 == 0, n)\n    flips = history(even)\n    for v in [6, 8, 9, 11, 12]:\n        n.value = v\n    assert flips == [False, True] and even.value is True\n    a = C(1)\n    stop = a.subscribe(lambda c: None)\n    stop()\n    a.value = 2\n    assert a.value == 2")

## Warm-up

`goal` doesn't know what a scoreboard, a commentary feed or a statistics counter is. It keeps a list of whoever asked to be told, and tells them. The statistics counter was added halfway through without touching `goal` at all. That's the Observer pattern in about ten lines.

## Who needs to know?

Here's the same feature written the direct way:

```python
def goal(team, minute):
    scoreboard.add(team, minute)
    commentary.shout(team)
    stats.count_goal(team)
    highlights.mark(minute)
```

`goal` is now coupled to four other parts and their method names (Week 34). Adding a fifth means editing `goal`, and testing `goal` means setting up all four. The parts that react depend on the event, but the event shouldn't depend on them.

## The Observer pattern

The Gang of Four describe Observer as: "define a one-to-many dependency between objects so that when one object changes state, all its dependents are notified and updated automatically." It has two roles:

- the **subject** (or publisher), which holds some state, keeps a list of observers, and notifies them when the state changes;
- the **observers** (or subscribers), which register with the subject and react when notified.

The classic form, with an observer interface:

Notice two choices in `Stock`:

- It notifies only when the price **actually changes** (setting 470 twice notified once).
- It sets the new price **before** notifying, so an observer that asks `stock.price` gets the new value.

The data is **pushed** to observers as arguments. The other style is **pull**: the subject passes itself (`o.update(self)`) and observers read what they need. Push keeps observers simpler; pull is useful when observers need different parts of a large state.

## Observers as functions

In Python, an observer with one method is just a function (yesterday's lesson again). The subject keeps a list of callables. A tidy addition, used by Week 33's `Signal`, is to have `subscribe` return a function that unsubscribes, so the caller doesn't need to keep the observer around just to remove it:

Any callable will do: a function, a lambda, a bound method (`display.price_changed`), or a callable object. The subject doesn't care.

## Named events

When a program has many kinds of event, a single **event bus** (sometimes called a dispatcher or message bus) can carry them all. Publishers announce an event by **name** with some data; subscribers register for the names they care about. Neither side needs to know the other exists:

`**data` passes the event's details as keyword arguments, and `**_` in a handler collects any it doesn't need, so new details can be added to an event without breaking older handlers. Many large programs (graphical interfaces, games, web frameworks) are organised around events like these.

## Pitfalls

Observer is simple, but it has well-known traps:

| Pitfall | What happens | Fix |
|---|---|---|
| notifying before updating | observers that read the subject see the old state | update first, then notify |
| an observer removes itself (or others) during notification | the loop over the list skips someone | loop over a **copy**: `for h in list(self._handlers)` |
| one observer raises an exception | the observers after it are never told | decide on purpose: let it stop everything, or catch, record and carry on |
| an observer changes the subject during notification | notifications nest inside each other, sometimes forever | avoid it, or guard against it |
| the order of observers matters | hidden coupling between observers | don't rely on order; if one must run first, call it directly |
| observers are never removed | the subject keeps them (and everything they refer to) alive | unsubscribe when done (Python's `weakref` module can also help) |

Here's the "one observer fails" case handled by catching and recording, which suits notifications where each observer is independent:

Catching `Exception` broadly is usually a mistake (Week 11), but here it's deliberate and the errors aren't hidden: they're returned so the caller can report them.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a program you use (a messaging app, a game, a music player) and list three events in it and the parts that probably observe each one. Tomorrow: the Factory pattern, for deciding which kind of object to create.